In [4]:
import pandas as pd
import numpy as np
import nltk
import re
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer

# Define stopwords
STOPWORDS = set(stopwords.words('english'))



In [5]:
# Remove special characters, punctuation, and convert to lowercase
def preprocess_text(text):
    if not isinstance(text, str):
        return ''  # Return empty string for NaN or non-string values
    text = re.sub(r'[^\w\s]', '', text.lower())  # Remove punctuation and convert to lowercase
    text = ' '.join(word for word in text.split() if word not in STOPWORDS)  # Remove stopwords
    return text

In [6]:


# Load the dataset
file_path = "filtered_file_2.xlsx"  # Replace with your actual file path
df = pd.read_excel(file_path)

# Apply text preprocessing
df['Abstract_clean'] = df['Abstract'].apply(preprocess_text)
df['Research_Areas_clean'] = df['Research Areas'].apply(preprocess_text)

# Combine text columns for analysis
df['combined_text'] = df['Abstract_clean'] + ' ' + df['Research_Areas_clean']

# Tokenize the text for FastText training
df['tokenized_text'] = df['combined_text'].apply(lambda x: x.split())



In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

def find_matching_researchers(project_idea: str, df: pd.DataFrame, top_n: int = 5) -> pd.DataFrame:
    """
    Find matching researchers using TF-IDF with logarithmic article count weighting.
    """
    print("Preprocessing data...")
    current_year = 2025
    
    # Pre-process researcher data
    researcher_records = []
    for _, row in df.iterrows():
        text = f"{row['Abstract_clean']} {row['Research_Areas_clean']} {row['Research_Areas_clean']} {row['Research_Areas_clean']}"
        researchers = [r.strip() for r in str(row['Researcher Ids']).split(';') if r.strip()]
        for researcher in researchers:
            researcher_records.append({
                'Researcher': researcher,
                'Text': text,
                'Publication_Year': row['Publication Year']
            })
    
    researcher_df = pd.DataFrame(researcher_records)
    
    print("Computing TF-IDF vectors...")
    vectorizer = TfidfVectorizer(
        max_features=2000,
        ngram_range=(1, 2),
        max_df=0.90,
        min_df=2,
        stop_words='english'
    )
    
    # Combine project idea with researcher texts
    all_texts = [project_idea] + researcher_df['Text'].tolist()
    tfidf_matrix = vectorizer.fit_transform(all_texts)
    
    print("Computing similarities...")
    similarities = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:]).flatten()
    
    # Add similarities to researcher DataFrame
    researcher_df['similarity'] = similarities
    
    # Group by researcher and compute metrics
    grouped = researcher_df.groupby('Researcher').agg({
        'similarity': 'max',
        'Publication_Year': 'max',
    }).reset_index()
    
    # Count articles per researcher
    article_counts = researcher_df['Researcher'].value_counts()
    grouped['article_count'] = grouped['Researcher'].map(article_counts)
    
    # Calculate normalized similarity using log of article count
    grouped['normalized_similarity'] = grouped['similarity'] * np.log10(1 + grouped['article_count'])
    
    # Calculate year weight
    grouped['year_weight'] = 1 / (1 + (np.log(1 + current_year - grouped['Publication_Year']) / np.log(1000)))
    
    # Calculate final score
    grouped['final_score'] = grouped['normalized_similarity'] * grouped['year_weight']
    
    # Get top matches
    result_df = grouped.nlargest(top_n, 'final_score')
    
    # Set index on Researcher column
    if 'Researcher' in result_df.columns:
        result_df.set_index('Researcher', inplace=True)
    else:
        raise KeyError("The 'Researcher' column is missing in the result DataFrame.")
    
    return result_df

In [ ]:
project_idea = "cell cancer bio bioinformatics machine learning algorithm human life gene gen heredity"

matches = find_matching_researchers(project_idea, df, top_n=10)

print("\nTop Recommended Collaborators:")
if not matches.empty and all(col in matches.columns for col in ['similarity', 'final_score', 'Publication_Year', 'article_count']):
	print(matches[['similarity', 'final_score', 'Publication_Year', 'article_count']])
else:
	print("No matching collaborators found or required columns are missing.")

# Search for a specific researcher
specific_researcher_id = 'ABG-9392-2021'
if not matches.empty and 'Researcher' in matches.columns:
	specific_researcher = matches[matches.index.str.contains(specific_researcher_id, case=False, na=False)]
else:
	specific_researcher = pd.DataFrame()

print("\nSpecific Researcher Details:")
if not specific_researcher.empty and all(col in specific_researcher.columns for col in ['similarity', 'final_score', 'Publication_Year', 'article_count']):
	print(specific_researcher[['similarity', 'final_score', 'Publication_Year', 'article_count']])
else:
	print("No details found for the specific researcher or required columns are missing.")

Preprocessing data...
Computing TF-IDF vectors...
